In [ ]:
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# Data Loading

In this step we load the datasets required for building the recommendation model.

Datasets used:

1. Attractions dataset  
   Contains places users can visit.

2. Hotels dataset  
   Used to recommend accommodation based on budget and location.

3. User interactions dataset  
   Simulated user behavior such as views, likes, or bookings.

4. Synthetic users dataset  
   Contains user profiles and interests.

These datasets will allow us to build a recommendation model that learns user preferences.

In [69]:
attractions = pd.read_csv("../data/processed/attractions_processed.csv")
hotels = pd.read_csv("../hotels.csv")
user_interactions = pd.read_csv("../data/processed/user_attraction_interactions.csv")
users = pd.read_csv("../data/processed/user_synthetic.csv")

In [70]:
print("Attractions:", attractions.shape)
print("Hotels:", hotels.shape)
print("Interactions:", user_interactions.shape)
print("Users:", users.shape)

Attractions: (386, 17)
Hotels: (3, 9)
Interactions: (1000000, 3)
Users: (5000, 9)


# Data Exploration

Before building the recommendation model, we need to understand the structure of the datasets.

We will inspect:

• dataset columns  
• missing values  
• data types  

This helps us decide which features can be used for the ML model.

In [71]:
print("Attractions Columns:")
print(attractions.columns)

print("\nHotels Columns:")
print(hotels.columns)

print("\nInteractions Columns:")
print(user_interactions.columns)

print("\nUsers Columns:")
print(users.columns)

Attractions Columns:
Index(['name', 'latitude', 'longitude', 'city', 'estimated_cost', 'category_attraction', 'category_museum', 'category_park', 'category_theme_park', 'attr_culture_score', 'attr_nature_score', 'attr_entertainment_score', 'interest_match_score', 'cost_ratio', 'budget_score', 'final_score', 'key'], dtype='str')

Hotels Columns:
Index(['name', 'city_name', 'latitude', 'longitude', 'price_per_night', 'rating', 'amenities', 'total_rooms', 'image_url'], dtype='str')

Interactions Columns:
Index(['user_id', 'attraction_name', 'interaction_score'], dtype='str')

Users Columns:
Index(['user_id', 'name', 'travel_style', 'budget_level', 'interest_culture', 'interest_nature', 'interest_food', 'interest_entertainment', 'outdoor_affinity'], dtype='str')


In [72]:
attractions.info()

<class 'pandas.DataFrame'>
RangeIndex: 386 entries, 0 to 385
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   name                      386 non-null    str    
 1   latitude                  386 non-null    float64
 2   longitude                 386 non-null    float64
 3   city                      386 non-null    str    
 4   estimated_cost            386 non-null    int64  
 5   category_attraction       386 non-null    int64  
 6   category_museum           386 non-null    int64  
 7   category_park             386 non-null    int64  
 8   category_theme_park       386 non-null    int64  
 9   attr_culture_score        386 non-null    int64  
 10  attr_nature_score         386 non-null    int64  
 11  attr_entertainment_score  386 non-null    int64  
 12  interest_match_score      386 non-null    float64
 13  cost_ratio                386 non-null    float64
 14  budget_score         

In [73]:
users.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   user_id                 5000 non-null   int64  
 1   name                    5000 non-null   str    
 2   travel_style            5000 non-null   str    
 3   budget_level            5000 non-null   str    
 4   interest_culture        5000 non-null   float64
 5   interest_nature         5000 non-null   float64
 6   interest_food           5000 non-null   float64
 7   interest_entertainment  5000 non-null   float64
 8   outdoor_affinity        5000 non-null   float64
dtypes: float64(5), int64(1), str(3)
memory usage: 351.7 KB


In [74]:
user_interactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 3 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   user_id            1000000 non-null  int64  
 1   attraction_name    1000000 non-null  str    
 2   interaction_score  1000000 non-null  float64
dtypes: float64(1), int64(1), str(1)
memory usage: 22.9 MB


# Build Training Dataset

To train the recommendation model we need to combine:

• User interactions  
• Attraction features  

Each row will represent:

User → Attraction → Interaction Score → Attraction Features

This dataset will be used to train the ranking model.

In [75]:
training_data = user_interactions.merge(
    attractions,
    left_on="attraction_name",
    right_on="name",
    how="left"
)

print(training_data.shape)
training_data.head()

(1057637, 20)


,user_id,attraction_name,interaction_score,name,latitude,longitude,city,estimated_cost,category_attraction,category_museum,category_park,category_theme_park,attr_culture_score,attr_nature_score,attr_entertainment_score,interest_match_score,cost_ratio,budget_score,final_score,key
0,1,Musée des Hospices Civils de Lyon,0.654494,Musée des Hospices Civils de Lyon,45.760841,4.831512,Lyon,15,0,1,0,0,1,0,0,0.780286,0.3,0.7,0.7562,1
1,1,Rivoli 59,0.653726,Rivoli 59,48.859227,2.345657,Paris,5,1,0,0,0,1,0,0,0.780286,0.1,0.9,0.8162,1
2,1,Point zéro des Routes de France,0.652487,Point zéro des Routes de France,48.853401,2.348788,Paris,5,1,0,0,0,1,0,0,0.780286,0.1,0.9,0.8162,1
3,1,Sortie des Catacombes,0.649706,Sortie des Catacombes,48.829526,2.334533,Paris,5,1,0,0,0,1,0,0,0.780286,0.1,0.9,0.8162,1
4,1,Pont du Carroussel,0.649333,Pont du Carroussel,48.859297,2.332888,Paris,5,1,0,0,0,1,0,0,0.780286,0.1,0.9,0.8162,1


In [76]:
training_data.isnull().sum().sort_values(ascending=False).head(10)

user_id                0
attraction_name        0
interaction_score      0
name                   0
latitude               0
longitude              0
city                   0
estimated_cost         0
category_attraction    0
category_museum        0
dtype: int64

# Add User Features

To improve recommendations we also include user characteristics.

These features represent user preferences such as:

• cultural interest  
• nature preference  
• entertainment interest  
• food preference  
• outdoor affinity  

These will allow the model to learn how different types of users interact with attractions.

In [77]:
training_data.columns

Index(['user_id', 'attraction_name', 'interaction_score', 'name', 'latitude', 'longitude', 'city', 'estimated_cost', 'category_attraction', 'category_museum', 'category_park', 'category_theme_park', 'attr_culture_score', 'attr_nature_score', 'attr_entertainment_score', 'interest_match_score', 'cost_ratio', 'budget_score', 'final_score', 'key'], dtype='str')

In [78]:
training_data = training_data.merge(
    users,
    on="user_id",
    how="left"
)
training_data["culture_match"] = (
    training_data["interest_culture"] *
    training_data["attr_culture_score"]
)

training_data["nature_match"] = (
    training_data["interest_nature"] *
    training_data["attr_nature_score"]
)

training_data["entertainment_match"] = (
    training_data["interest_entertainment"] *
    training_data["attr_entertainment_score"]
)

print(training_data.shape)
training_data.head()

(1057637, 31)


,user_id,attraction_name,interaction_score,name_x,latitude,longitude,city,estimated_cost,category_attraction,category_museum,category_park,category_theme_park,attr_culture_score,attr_nature_score,attr_entertainment_score,interest_match_score,cost_ratio,budget_score,final_score,key,name_y,travel_style,budget_level,interest_culture,interest_nature,interest_food,interest_entertainment,outdoor_affinity,culture_match,nature_match,entertainment_match
0,1,Musée des Hospices Civils de Lyon,0.654494,Musée des Hospices Civils de Lyon,45.760841,4.831512,Lyon,15,0,1,0,0,1,0,0,0.780286,0.3,0.7,0.7562,1,Samantha Garrett,Backpacker,Low,0.780286,0.892798,0.539463,0.346806,0.746798,0.780286,0.0,0.0
1,1,Rivoli 59,0.653726,Rivoli 59,48.859227,2.345657,Paris,5,1,0,0,0,1,0,0,0.780286,0.1,0.9,0.8162,1,Samantha Garrett,Backpacker,Low,0.780286,0.892798,0.539463,0.346806,0.746798,0.780286,0.0,0.0
2,1,Point zéro des Routes de France,0.652487,Point zéro des Routes de France,48.853401,2.348788,Paris,5,1,0,0,0,1,0,0,0.780286,0.1,0.9,0.8162,1,Samantha Garrett,Backpacker,Low,0.780286,0.892798,0.539463,0.346806,0.746798,0.780286,0.0,0.0
3,1,Sortie des Catacombes,0.649706,Sortie des Catacombes,48.829526,2.334533,Paris,5,1,0,0,0,1,0,0,0.780286,0.1,0.9,0.8162,1,Samantha Garrett,Backpacker,Low,0.780286,0.892798,0.539463,0.346806,0.746798,0.780286,0.0,0.0
4,1,Pont du Carroussel,0.649333,Pont du Carroussel,48.859297,2.332888,Paris,5,1,0,0,0,1,0,0,0.780286,0.1,0.9,0.8162,1,Samantha Garrett,Backpacker,Low,0.780286,0.892798,0.539463,0.346806,0.746798,0.780286,0.0,0.0


In [79]:
training_data.isnull().sum().sort_values(ascending=False).head(10)

user_id                0
attraction_name        0
interaction_score      0
name_x                 0
latitude               0
longitude              0
city                   0
estimated_cost         0
category_attraction    0
category_museum        0
dtype: int64

# Feature Selection

Before training the model we select the features that will be used for learning.

We remove columns that are identifiers or text fields such as:

• attraction names  
• user names  
• city names  

The remaining numerical columns represent user preferences and attraction characteristics.

These will be used as input features for the recommendation model.

In [80]:
features = [

# attraction features
"estimated_cost",
"category_attraction",
"category_museum",
"category_park",
"category_theme_park",

"attr_culture_score",
"attr_nature_score",
"attr_entertainment_score",

"cost_ratio",
"budget_score",

# user features
"interest_culture",
"interest_nature",
"interest_food",
"interest_entertainment",
"outdoor_affinity",

# NEW MATCH FEATURES
"culture_match",
"nature_match",
"entertainment_match"

]

In [81]:
X = training_data[features]

y = training_data["interaction_score"]

print(X.shape)

(1057637, 18)


# Ranking Groups

We are training a ranking model, which means the model must learn how to order attractions for each user.

To do this we create groups based on user_id.

Each group represents all attractions interacted with by the same user.

In [82]:
groups = training_data.groupby("user_id").size()

print(groups.head())
print("Total groups:", len(groups))

group_list = groups.tolist()

print("Number of groups:", len(group_list))

user_id
1    213
2    208
3    213
4    214
5    214
dtype: int64
Total groups: 5000
Number of groups: 5000


# Train Test Split

Before training the ranking model we split the dataset into:

• Training set  
• Test set  

The training set is used for learning patterns.

The test set is used to evaluate how well the model generalizes.

In [83]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (846109, 18)
Test shape: (211528, 18)


# Train Ranking Model

We train a LightGBM ranking model to learn how to order attractions for each user.

The model will learn relationships between:

• user preferences  
• attraction characteristics  

and predict which attractions should be ranked higher for a given user.

In [85]:
# Create ranking groups (each user is a group)
groups = training_data.groupby("user_id").size()

# Convert to list for LightGBM
group_list = groups.tolist()

print("Total groups:", len(group_list))

Total groups: 5000


In [86]:
import lightgbm as lgb

model = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    boosting_type="gbdt",
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

model.fit(
    X,              # feature matrix
    y_rank,         # ranking labels
    group=group_list
)

print("Model trained successfully")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.052973 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1805
[LightGBM] [Info] Number of data points in the train set: 1057637, number of used features: 15
Model trained successfully


# Convert Interaction Scores to Ranking Labels

Ranking models require integer relevance labels instead of continuous values.

We convert interaction_score into discrete relevance levels so the model can learn ranking priorities.

In [87]:
y_rank = pd.cut(
    training_data["interaction_score"],
    bins=[0, 0.25, 0.5, 0.75, 1],
    labels=[0, 1, 2, 3]
).astype(int)

print(y_rank.head())

0    2
1    2
2    2
3    2
4    2
Name: interaction_score, dtype: int64


In [88]:
y_rank.value_counts()

interaction_score
2    1004252
3      53385
Name: count, dtype: int64

# Train Ranking Model

We train a LightGBM ranking model using grouped data.

Each group represents a single user, and the model learns how to rank attractions within each user's interactions.

In [89]:
import lightgbm as lgb

model = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    boosting_type="gbdt",
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

model.fit(
    X,
    y_rank,
    group=group_list
)

print("Model trained successfully")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.036660 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1805
[LightGBM] [Info] Number of data points in the train set: 1057637, number of used features: 15
Model trained successfully


# Testing the Recommendation Model

In this step we simulate how the website will use the trained model.

We select a user and predict recommendation scores for attractions.

The model will rank attractions based on user preferences and attraction features.

In [90]:
user_id_test = 25


user_data = training_data[training_data["user_id"] == user_id_test].copy()

print(user_data.shape)

(214, 31)


In [91]:
X_user = user_data[features]

In [92]:
user_data["predicted_score"] = model.predict(X_user)

In [ ]:
recommendations = user_data.sort_values(
    "predicted_score",
    ascending=False
)

recommendations[
    ["attraction_name", "city", "predicted_score"]
].head(10)

,attraction_name,city,predicted_score
5084,Les Bateaux Lyonnais,Lyon,-4.522411
5085,Tapir malais,Paris,-4.522411
5082,Point zéro des Routes de France,Paris,-4.522411
5081,Pont du Carroussel,Paris,-4.522411
5080,Sortie des Catacombes,Paris,-4.522411
5079,Rivoli 59,Paris,-4.522411
5090,Japon - Chine,Paris,-4.522411
5091,Hôtel d'Estaing,Lyon,-4.522411
5088,Martre à gorge jaune,Paris,-4.522411
5089,Caucase,Paris,-4.522411


In [93]:
import pandas as pd

importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
})

importance.sort_values(
    "importance",
    ascending=False
)

,feature,importance
15,culture_match,1562
10,interest_culture,1337
13,interest_entertainment,1326
11,interest_nature,878
12,interest_food,529
0,estimated_cost,196
14,outdoor_affinity,168
1,category_attraction,4
2,category_museum,0
3,category_park,0


In [94]:
import joblib

joblib.dump(model, "travel_recommender_model.pkl")

print("Model saved successfully")

Model saved successfully


In [95]:
budget_config = {
    "low": {"daily_budget":100,"hotel_ratio":0.6,"attraction_ratio":0.4},
    "medium": {"daily_budget":200,"hotel_ratio":0.6,"attraction_ratio":0.4},
    "high": {"daily_budget":400,"hotel_ratio":0.6,"attraction_ratio":0.4}
}

In [96]:
def calculate_budget_split(budget_level):

    config = budget_config[budget_level.lower()]

    daily_budget = config["daily_budget"]

    hotel_budget = daily_budget * config["hotel_ratio"]

    attraction_budget = daily_budget * config["attraction_ratio"]

    return hotel_budget, attraction_budget

In [97]:
calculate_budget_split("low")

(60.0, 40.0)

In [98]:
def select_hotel(city, hotel_budget):

    city_hotels = hotels[
        hotels["city_name"] == city
    ].copy()

    # Filter by price
    filtered = city_hotels[
        city_hotels["price_per_night"] <= hotel_budget
    ]

    # If no hotel in budget → choose cheapest
    if filtered.empty:

        city_hotels = city_hotels.sort_values(
            "price_per_night"
        )

        return city_hotels.head(1)

    # Otherwise choose best rated
    filtered = filtered.sort_values(
        "rating",
        ascending=False
    )

    return filtered.head(1)

In [99]:
def generate_trip_itinerary(
    user_id,
    city,
    trip_days,
    attractions_per_day,
    budget_level
):

    hotel_budget, attraction_budget = calculate_budget_split(budget_level)

    hotel = select_hotel(city, hotel_budget)

    city_data = training_data[
        (training_data["user_id"] == user_id) &
        (training_data["city"] == city) &
        (training_data["estimated_cost"] <= attraction_budget)
    ].copy()

    city_data = city_data[
        city_data["category_attraction"] == 1
    ]

    if city_data.empty:
        city_data = training_data[
            (training_data["user_id"] == user_id) &
            (training_data["city"] == city)
        ].copy()

    X_user = city_data[features]

    city_data["predicted_score"] = model.predict(X_user)

    ranked = city_data.sort_values(
        "predicted_score",
        ascending=False
    )

    total_needed = trip_days * attractions_per_day

    top_places = ranked.head(total_needed)

    itinerary = {}

    places = top_places["attraction_name"].tolist()

    index = 0

    for day in range(1, trip_days + 1):

        itinerary[f"Day {day}"] = places[index:index+attractions_per_day]

        index += attractions_per_day

    hotel_name = hotel.iloc[0]["name"]
    hotel_price = hotel.iloc[0]["price_per_night"]

    return {
        "hotel": hotel_name,
        "hotel_price": hotel_price,
        "itinerary": itinerary
    }

In [100]:
generate_trip_itinerary(
    user_id=25,
    city="Paris",
    trip_days=3,
    attractions_per_day=4,
    budget_level="low"
)

{'hotel': 'Hilton Paris',
 'hotel_price': np.int64(180),
 'itinerary': {'Day 1': ['Rivoli 59',
   'Sortie des Catacombes',
   'Pont du Carroussel',
   'Point zéro des Routes de France'],
  'Day 2': ['Tapir malais',
   'Binturong',
   'Plaque commémorant le bombardement du allemand «  canons parisiens » le 12 avril 1918',
   'Martre à gorge jaune'],
  'Day 3': ['Caucase', 'Japon - Chine', 'Corse', 'Chat de Pallas']}}

# Trip Generation Engine

This module converts ranked attractions into a daily travel itinerary.

The algorithm selects top ranked attractions and distributes them across trip days.

# Trip Request Input

This structure represents the parameters provided by the user when generating a trip.

The parameters include:

- city
- number of days
- attractions per day
- budget level
- user id

In [142]:
trip_request = {
    "user_id": 25,
    "city": "Paris",
    "trip_days": 3,
    "attractions_per_day": 4,
    "budget_level": "low"
}

# Extract Trip Parameters

We extract the values from the trip request so they can be used in the itinerary generation pipeline.

In [143]:
user_id = trip_request["user_id"]
city = trip_request["city"]
trip_days = trip_request["trip_days"]
attractions_per_day = trip_request["attractions_per_day"]
budget_level = trip_request["budget_level"]

# Ranking Attractions

In this step we generate personalized attraction recommendations for a user.

The process includes:

• Filtering attractions by user and selected city  
• Removing invalid entries such as zoo animals from the dataset  
• Using the trained LightGBM ranking model to predict recommendation scores  
• Sorting attractions by predicted score to identify the most relevant places

The top ranked attractions are then used as candidates for trip generation and location clustering.

In [158]:
# Select user and city
user_id_test = 25
city = "Paris"

# Filter attractions for this user and city
user_data = training_data[
    (training_data["user_id"] == user_id_test) &
    (training_data["city"] == city)
].copy()

# Remove zoo / animal entries
animal_keywords = [
    "tapir","binturong","oryx","martre",
    "amazone","caucase","émeu","chat",
    "renard","chèvres"
]

user_data = user_data[
    ~user_data["attraction_name"].str.lower().str.contains(
        "|".join(animal_keywords),
        na=False
    )
]

# Build feature matrix
X_user = user_data[features]

# Predict recommendation scores
user_data["predicted_score"] = model.predict(X_user)

# Rank attractions
ranked_attractions = user_data.sort_values(
    "predicted_score",
    ascending=False
)

# Select candidate attractions
top_places = ranked_attractions.head(20)

top_places.head(10)

,user_id,attraction_name,interaction_score,name_x,latitude,longitude,city,estimated_cost,category_attraction,category_museum,category_park,category_theme_park,attr_culture_score,attr_nature_score,attr_entertainment_score,interest_match_score,cost_ratio,budget_score,final_score,key,name_y,travel_style,budget_level,interest_culture,interest_nature,interest_food,interest_entertainment,outdoor_affinity,culture_match,nature_match,entertainment_match,predicted_score
5080,25,Sortie des Catacombes,0.571587,Sortie des Catacombes,48.829526,2.334533,Paris,5,1,0,0,0,1,0,0,0.780286,0.1,0.9,0.8162,1,Erik Moore,Backpacker,Low,0.414755,0.843826,0.501072,0.315444,0.783594,0.414755,0.0,0.0,-4.522411
5079,25,Rivoli 59,0.572687,Rivoli 59,48.859227,2.345657,Paris,5,1,0,0,0,1,0,0,0.780286,0.1,0.9,0.8162,1,Erik Moore,Backpacker,Low,0.414755,0.843826,0.501072,0.315444,0.783594,0.414755,0.0,0.0,-4.522411
5090,25,Japon - Chine,0.561901,Japon - Chine,48.844097,2.359489,Paris,5,1,0,0,0,1,0,0,0.780286,0.1,0.9,0.8162,1,Erik Moore,Backpacker,Low,0.414755,0.843826,0.501072,0.315444,0.783594,0.414755,0.0,0.0,-4.522411
5087,25,Plaque commémorant le bombardement du allemand...,0.564439,Plaque commémorant le bombardement du allemand...,48.855596,2.359932,Paris,5,1,0,0,0,1,0,0,0.780286,0.1,0.9,0.8162,1,Erik Moore,Backpacker,Low,0.414755,0.843826,0.501072,0.315444,0.783594,0.414755,0.0,0.0,-4.522411
5081,25,Pont du Carroussel,0.571447,Pont du Carroussel,48.859297,2.332888,Paris,5,1,0,0,0,1,0,0,0.780286,0.1,0.9,0.8162,1,Erik Moore,Backpacker,Low,0.414755,0.843826,0.501072,0.315444,0.783594,0.414755,0.0,0.0,-4.522411
5082,25,Point zéro des Routes de France,0.569764,Point zéro des Routes de France,48.853401,2.348788,Paris,5,1,0,0,0,1,0,0,0.780286,0.1,0.9,0.8162,1,Erik Moore,Backpacker,Low,0.414755,0.843826,0.501072,0.315444,0.783594,0.414755,0.0,0.0,-4.522411
5092,25,Corse,0.558950,Corse,48.844159,2.359014,Paris,5,1,0,0,0,1,0,0,0.780286,0.1,0.9,0.8162,1,Erik Moore,Backpacker,Low,0.414755,0.843826,0.501072,0.315444,0.783594,0.414755,0.0,0.0,-4.522411
5096,25,Pendule de Foucault,0.557536,Pendule de Foucault,48.846193,2.346136,Paris,5,1,0,0,0,1,0,0,0.780286,0.1,0.9,0.8162,1,Erik Moore,Backpacker,Low,0.414755,0.843826,0.501072,0.315444,0.783594,0.414755,0.0,0.0,-4.522411
5095,25,Résidence Rollin,0.557564,Résidence Rollin,48.844808,2.350314,Paris,5,1,0,0,0,1,0,0,0.780286,0.1,0.9,0.8162,1,Erik Moore,Backpacker,Low,0.414755,0.843826,0.501072,0.315444,0.783594,0.414755,0.0,0.0,-4.522411
5094,25,Outardes Houbara,0.557824,Outardes Houbara,48.846309,2.360518,Paris,5,1,0,0,0,1,0,0,0.780286,0.1,0.9,0.8162,1,Erik Moore,Backpacker,Low,0.414755,0.843826,0.501072,0.315444,0.783594,0.414755,0.0,0.0,-4.522411


# Selecting Candidate Attractions

We select a pool of top attractions before clustering them into daily groups.

In [159]:
total_needed = trip_days * attractions_per_day

top_places = ranked_attractions.head(total_needed * 2)

# Preparing Location Data

Latitude and longitude are used to cluster attractions geographically.

In [160]:
location_data = top_places[["latitude", "longitude"]]

# Location Clustering

K-Means clustering groups attractions into clusters corresponding to travel days.

In [161]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=trip_days,
    random_state=42
)

top_places["cluster"] = kmeans.fit_predict(location_data)

# Generating Daily Itinerary

Each cluster corresponds to a travel day.
Attractions inside each cluster are sorted by predicted recommendation score.

In [172]:
# Select hotels for the chosen city
city_hotels = hotels[hotels["city_name"] == city].copy()

# Sort by rating (best first)
city_hotels = city_hotels.sort_values("rating", ascending=False)

# Pick the best hotel
selected_hotel = city_hotels.iloc[0]

hotel_name = selected_hotel["name"]
hotel_price = selected_hotel["price_per_night"]

In [173]:
from pprint import pprint

itinerary = {}
remaining = []

for day in range(trip_days):

    day_places = top_places[
        top_places["cluster"] == day
    ]

    day_places = day_places.sort_values(
        "predicted_score",
        ascending=False
    )

    selected = day_places.head(attractions_per_day)

    # store selected attractions
    itinerary[f"Day {day+1}"] = selected[
        ["attraction_name", "estimated_cost"]
    ].to_dict("records")

    # store leftover attractions (as dictionaries)
    leftover = day_places.iloc[attractions_per_day:]

    remaining.extend(
        leftover[["attraction_name", "estimated_cost"]]
        .to_dict("records")
    )


for day in range(1, trip_days + 1):

    while len(itinerary[f"Day {day}"]) < attractions_per_day and remaining:

        itinerary[f"Day {day}"].append(remaining.pop(0))


pprint(itinerary)

{'Day 1': [{'attraction_name': 'Sortie des Catacombes', 'estimated_cost': 5},
           {'attraction_name': 'Japon - Chine', 'estimated_cost': 5},
           {'attraction_name': 'Plaque commémorant le bombardement du allemand '
                               '«  canons parisiens » le 12 avril 1918',
            'estimated_cost': 5},
           {'attraction_name': 'Corse', 'estimated_cost': 5}],
 'Day 2': [{'attraction_name': 'Marché aux Timbres', 'estimated_cost': 5},
           {'attraction_name': 'Pendule de Foucault', 'estimated_cost': 5},
           {'attraction_name': 'Résidence Rollin', 'estimated_cost': 5},
           {'attraction_name': 'Outardes Houbara', 'estimated_cost': 5}],
 'Day 3': [{'attraction_name': 'Rivoli 59', 'estimated_cost': 5},
           {'attraction_name': 'Pont du Carroussel', 'estimated_cost': 5},
           {'attraction_name': 'Point zéro des Routes de France',
            'estimated_cost': 5},
           {'attraction_name': 'Pendule de Foucault', 'estimat

In [175]:
print(f"Hotel: {hotel_name} (€{hotel_price} per night)")
print()

for day, places in itinerary.items():

    print(day)

    attraction_cost = 0

    for p in places:

        name = p["attraction_name"]
        price = p["estimated_cost"]

        print(f" - {name} (€{price})")

        attraction_cost += price

    total_daily = hotel_price + attraction_cost

    print()
    print(f"   Attraction cost: €{attraction_cost}")
    print(f"   Hotel cost: €{hotel_price}")
    print(f"   Total daily cost: €{total_daily}")
    print()

Hotel: Hilton Paris (€180 per night)

Day 1
 - Sortie des Catacombes (€5)
 - Japon - Chine (€5)
 - Plaque commémorant le bombardement du allemand «  canons parisiens » le 12 avril 1918 (€5)
 - Corse (€5)

   Attraction cost: €20
   Hotel cost: €180
   Total daily cost: €200

Day 2
 - Marché aux Timbres (€5)
 - Pendule de Foucault (€5)
 - Résidence Rollin (€5)
 - Outardes Houbara (€5)

   Attraction cost: €20
   Hotel cost: €180
   Total daily cost: €200

Day 3
 - Rivoli 59 (€5)
 - Pont du Carroussel (€5)
 - Point zéro des Routes de France (€5)
 - Pendule de Foucault (€5)

   Attraction cost: €20
   Hotel cost: €180
   Total daily cost: €200



# Trip Generation Pipeline

This function generates a complete travel plan by combining:

• ML attraction ranking  
• Location clustering (KMeans)  
• Hotel recommendation  
• Budget estimation  

The function returns a structured trip plan that can be used directly by the backend API.

In [183]:
def generate_trip(user_id, city, trip_days, attractions_per_day):

    # -----------------------------
    # 1 Filter user + city
    # -----------------------------

    user_data = training_data[
        (training_data["user_id"] == user_id) &
        (training_data["city"] == city)
    ].copy()

    # -----------------------------
    # 2 Predict attraction scores
    # -----------------------------

    X_user = user_data[features]

    user_data["predicted_score"] = model.predict(X_user)

    ranked_attractions = user_data.sort_values(
        "predicted_score",
        ascending=False
    )

    total_needed = trip_days * attractions_per_day

    top_places = ranked_attractions.head(total_needed * 2)

    # -----------------------------
    # 3 Location clustering
    # -----------------------------

    from sklearn.cluster import KMeans

    location_data = top_places[["latitude", "longitude"]]

    kmeans = KMeans(
        n_clusters=trip_days,
        random_state=42
    )

    top_places["cluster"] = kmeans.fit_predict(location_data)

    # -----------------------------
    # 4 Build itinerary
    # -----------------------------

    itinerary = {}
    remaining = []

    for day in range(trip_days):

        day_places = top_places[
            top_places["cluster"] == day
        ]

        day_places = day_places.sort_values(
            "predicted_score",
            ascending=False
        )

        selected = day_places.head(attractions_per_day)

        oredered = order_by_distance(selected)

        selected = pd.DataFrame(oredered)

        itinerary[f"Day {day+1}"] = selected[
            ["attraction_name", "estimated_cost"]
        ].to_dict("records")

        leftover = day_places.iloc[attractions_per_day:]

        remaining.extend(
            leftover[["attraction_name", "estimated_cost"]]
            .to_dict("records")
        )

    for day in range(1, trip_days + 1):

        while len(itinerary[f"Day {day}"]) < attractions_per_day and remaining:

            itinerary[f"Day {day}"].append(remaining.pop(0))

    # -----------------------------
    # 5 Select hotel automatically
    # -----------------------------

    city_hotels = hotels[
        hotels["city_name"] == city
    ].copy()

    city_hotels = city_hotels.sort_values(
        "rating",
        ascending=False
    )

    selected_hotel = city_hotels.iloc[0]

    hotel_info = {
        "name": selected_hotel["name"],
        "price_per_night": int(selected_hotel["price_per_night"])
    }

    # -----------------------------
    # 6 Return final trip
    # -----------------------------

    return {
        "hotel": hotel_info,
        "itinerary": itinerary
    }

In [184]:
trip = generate_trip(
    user_id=25,
    city="Paris",
    trip_days=3,
    attractions_per_day=4
)

from pprint import pprint
pprint(trip)

{'hotel': {'name': 'Hilton Paris', 'price_per_night': 180},
 'itinerary': {'Day 1': [{'attraction_name': 'Binturong', 'estimated_cost': 5},
                         {'attraction_name': 'Amazone à joues vertes',
                          'estimated_cost': 5},
                         {'attraction_name': 'Tapir malais',
                          'estimated_cost': 5},
                         {'attraction_name': 'Voiture RATP',
                          'estimated_cost': 5}],
               'Day 2': [{'attraction_name': 'Rivoli 59', 'estimated_cost': 5},
                         {'attraction_name': 'Point zéro des Routes de France',
                          'estimated_cost': 5},
                         {'attraction_name': 'Pendule de Foucault',
                          'estimated_cost': 5},
                         {'attraction_name': 'Pont du Carroussel',
                          'estimated_cost': 5}],
               'Day 3': [{'attraction_name': 'Sortie des Catacombes',
            

# Distance-Based Attraction Ordering

After clustering attractions by location, we further improve the itinerary by ordering attractions based on geographic distance.

This ensures that attractions within the same day are visited in a logical walking route rather than random order.

In [185]:
from math import radians, sin, cos, sqrt, atan2

def haversine_distance(lat1, lon1, lat2, lon2):

    R = 6371  # Earth radius in km

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))

    return R * c

In [186]:
def order_by_distance(day_places):

    if len(day_places) <= 1:
        return day_places

    ordered = [day_places.iloc[0]]
    remaining = day_places.iloc[1:].copy()

    while len(remaining) > 0:

        last = ordered[-1]

        remaining["distance"] = remaining.apply(
            lambda row: haversine_distance(
                last["latitude"],
                last["longitude"],
                row["latitude"],
                row["longitude"]
            ),
            axis=1
        )

        next_place = remaining.sort_values(
            "distance"
        ).iloc[0]

        ordered.append(next_place)

        remaining = remaining.drop(next_place.name)

    return ordered

In [187]:
import joblib

joblib.dump(model, "../app/ml/travel_ranker.pkl")

print("Model saved successfully")

Model saved successfully
